# Inventarios 360

## Inicio del proyecto BI

Este notebook inicia el trabajo analítico sobre el archivo `Fechas de Vencimiento Beauty Care 1.xlsx`.

En esta primera versión se implementan dos reglas base del proyecto:

- identificador compuesto del producto por `Item ID + Contenedor`;
- cálculo de `dias_en_inventario` usando una `fecha_corte` explícita menos `Fecha de Ingreso`.


## 1. Imports y configuración

Si `openpyxl` no está instalado en el entorno, ejecutar primero la instalación.


In [ ]:
# Inventarios 360 — Configuración del entorno
import sys
from pathlib import Path

# Asegurar que src/ esté en el path
sys.path.insert(0, str(Path.cwd()))

from dotenv import load_dotenv
load_dotenv()

from src.etl import FECHA_CORTE, UMBRAL_CRITICO, UMBRAL_PROXIMO
print(f"fecha_corte    : {FECHA_CORTE}")
print(f"umbral_critico : {UMBRAL_CRITICO} días")
print(f"umbral_proximo : {UMBRAL_PROXIMO} días")

In [ ]:
from src.etl.pipeline import run_pipeline

# load_to_db=True requiere variables de entorno configuradas en .env
df_clean, df_dropped = run_pipeline(load_to_db=False)
df_clean.head()

In [ ]:
import pandas as pd

print(f"Shape: {df_clean.shape}")
print(f"\nTipos:")
print(df_clean.dtypes)
print(f"\nNulos por columna:")
print(df_clean.isnull().sum())
print(f"\nDistribución de calidad_flag:")
print(df_clean["calidad_flag"].value_counts())
print(f"\nDistribución de estado_inventario:")
print(df_clean["estado_inventario"].value_counts())
print(f"\nRango de score_riesgo: {df_clean['score_riesgo'].min()} – {df_clean['score_riesgo'].max()}")

In [ ]:
# %pip install openpyxl

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

fecha_corte = pd.Timestamp("2026-04-10")
data_path = Path("datos_proyecto") / "Fechas de Vencimiento Beauty Care 1.xlsx"
sheet_name = "Page1_1"

print(f"Archivo: {data_path}")
print(f"Hoja: {sheet_name}")
print(f"Fecha de corte: {fecha_corte.date()}")


## 2. Carga de datos

Se carga la hoja principal del inventario y se hace una normalización inicial de columnas.


In [ ]:
df_raw = pd.read_excel(data_path, sheet_name=sheet_name)
df_raw.columns = df_raw.columns.str.strip()

print(df_raw.shape)
df_raw.head()


## 3. Estandarización mínima

En esta etapa se convierten las columnas críticas a tipos utilizables para análisis.


In [ ]:
df = df_raw.copy()

df["Item ID"] = df["Item ID"].astype("string").str.strip()
df["Contenedor"] = df["Contenedor"].astype("string").str.strip()
df["Unds"] = pd.to_numeric(df["Unds"], errors="coerce")
df["Fecha de Ingreso"] = pd.to_datetime(df["Fecha de Ingreso"], errors="coerce")
df["Fecha Vencimiento"] = pd.to_datetime(df["Fecha Vencimiento"], errors="coerce")

df[["Item ID", "Contenedor", "Unds", "Fecha de Ingreso", "Fecha Vencimiento"]].head()


## 4. Identificador compuesto

El identificador operativo del registro se construye con `Item ID + Contenedor`.


In [ ]:
df["product_container_id"] = (
    df["Item ID"].fillna("SIN_ITEM").astype("string").str.strip()
    + "_"
    + df["Contenedor"].fillna("SIN_CONTENEDOR").astype("string").str.strip()
)

df[["Item ID", "Contenedor", "product_container_id"]].head()


## 5. Antigüedad en inventario

Se calcula cuántos días lleva el inventario almacenado usando la fecha de corte definida para el proyecto.


In [ ]:
df["dias_en_inventario"] = (fecha_corte - df["Fecha de Ingreso"]).dt.days
df["dias_para_vencimiento"] = (df["Fecha Vencimiento"] - fecha_corte).dt.days

df[[
    "product_container_id",
    "Fecha de Ingreso",
    "Fecha Vencimiento",
    "dias_en_inventario",
    "dias_para_vencimiento",
]].head()


## 6. Validación rápida

Se revisa el resultado de las variables nuevas para confirmar que ya existe una base mínima para análisis de riesgo.


In [ ]:
df[["dias_en_inventario", "dias_para_vencimiento", "Unds"]].describe(include="all")


In [ ]:
df[[
    "Marca",
    "Item ID",
    "Descripción",
    "Contenedor",
    "product_container_id",
    "dias_en_inventario",
    "dias_para_vencimiento",
    "Unds",
]].head(10)
